In [1]:
from spacerocks import SpaceRock
from spacerocks.time import Time
from spacerocks.observing import Observatory, Observation
from spacerocks.spice import SpiceKernel
from spacerocks.nbody import Simulation, Force
import numpy as np


import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

kernel = SpiceKernel()
kernel.load("/Users/kjnapier/data/spice/latest_leapseconds.tls")
kernel.load("/Users/kjnapier/data/spice/sb441-n16.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-1.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-2.bsp")
kernel.load("/Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc")

Loading kernel: /Users/kjnapier/data/spice/latest_leapseconds.tls
Loading kernel: /Users/kjnapier/data/spice/sb441-n16.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-1.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-2.bsp
Loading kernel: /Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc


In [2]:
w84 = Observatory.from_obscode('w84')

In [3]:
epoch = Time.now()


rock = SpaceRock.from_horizons("arrokoth", epoch=epoch, origin="ssb", reference_plane="J2000")
sim = Simulation.horizons(epoch, "J2000", "ssb")
sim.add(rock)

In [4]:
observations = []    
for idx in range(0, 300, 30):
    sim.integrate(epoch + idx)
    observer = w84.at(epoch + idx, reference_plane="J2000", origin="ssb")
    rock = sim.get_particle("arrokoth")
    obs = rock.observe(observer)
    observations.append(obs)

In [5]:
smear = 0.2/3600 * (np.pi / 180)

In [6]:
smear

9.69627362219072e-07

In [7]:
simulated_observations = []
for obs in observations:
    ra = obs.ra
    dec = obs.dec
    ra += np.random.normal(0, smear)
    dec += np.random.normal(0, smear)
    epoch = obs.epoch
    observer = obs.observer
    cov = [[smear**2, 0], [0, smear**2]]
    simulated_o = Observation.from_astrometry(obs.epoch, ra, dec, obs.observer)
    simulated_o.set_covariance(cov)
    simulated_observations.append(simulated_o)

In [8]:
from spacerocks.orbfit import gauss, fit_orbit_lm, randj

In [9]:
rocks = gauss(simulated_observations[0], simulated_observations[5], simulated_observations[9], min_distance=1e-6)

In [10]:
for rock in rocks:
    print(rock.a(), rock.e(), rock.inc())

44.17592578183586 0.025278522797804393 0.36947129726665334


In [11]:
randj(simulated_observations, rock, sim)

(VecStorage { data: [0.3430446286537989, 3.7406979145748354, 3.7503622235122243, 4.963886553083796, 2.314915554073508, 0.399917156648308, 1.8176619266288714, 2.3711330657770504, 1.1905700243457693, 0.11173499237113062], nrows: Dyn(10), ncols: Const }, VecStorage { data: [-22597.982280714303, 22575.324335560865, 19755.801839660413, 20346.456957387458, 6274.907859182832, 23563.98269152127, -12760.590330906929, -22107.828437167853, -8008.318583663243, -23089.34205051544, -9891.235442860412, 10632.57942439666, 4548.889634437003, 5034.837513129275, -5819.824226493964, 10422.841421983687, -12997.286514615513, -12330.897371937155, 5019.1999403037535, -9724.907162205614, -3034.4041782620625, 1957.826285314823, 14189.118022245317, 13772.362493069679, 22858.71232359238, 3198.9615723593265, 17338.366268482285, 4762.281380532585, -22193.609240750422, -2970.671424420285, 3389907.302593695, -2709168.6913919942, -1778047.4871703645, -1220747.8100060064, -188218.60511324395, 136.811894671518, -382886.

In [12]:
sim = Simulation.giants(rocks[0].epoch, "J2000", "ssb")

In [13]:
rocks[0].position

(17.882605377410506, -36.456117905089656, -14.39778317001854)

In [14]:
fit_orbit_lm(simulated_observations, rocks[0], sim)

Iteration: 0, chisq: 21.003920088475347, lambda: 0.01, ndof: 14
Iteration: 1, chisq: 13.692986800196383, lambda: 0.0011111111111111111, ndof: 14
Iteration: 2, chisq: 13.692986800196383, lambda: 0.012222222222222223, ndof: 14
Iteration: 3, chisq: 13.692986800196383, lambda: 0.13444444444444445, ndof: 14
Iteration: 4, chisq: 13.692986800196383, lambda: 1.478888888888889, ndof: 14
Iteration: 5, chisq: 13.692986800196383, lambda: 16.267777777777777, ndof: 14
Iteration: 6, chisq: 13.692986800196383, lambda: 178.94555555555556, ndof: 14
Iteration: 7, chisq: 13.692986800196383, lambda: 1968.401111111111, ndof: 14
Iteration: 8, chisq: 13.692986800196383, lambda: 21652.41222222222, ndof: 14
Iteration: 9, chisq: 13.692986800196383, lambda: 238176.53444444443, ndof: 14
Iteration: 10, chisq: 13.692986800196383, lambda: 2619941.8788888888, ndof: 14
Iteration: 11, chisq: 13.692986800196383, lambda: 28819360.667777777, ndof: 14
Iteration: 12, chisq: 13.692986800196383, lambda: 317012967.34555554, ndo

In [15]:
randj(simulated_observations, rocks[0], sim)

(VecStorage { data: [0.37171047506083643, 23.215107114409125, 28.081144684452802, 23.303708073722774, 11.773757388609706, 0.4001898567637691, 6.720831630716421, 6.262654473650706, 0.3991513412204933, 0.08880243171795763], nrows: Dyn(10), ncols: Const }, VecStorage { data: [-22543.675540986907, -22566.089701570036, -22750.437192797788, -22977.80785447401, -22721.164726391406, 23547.203918666826, 23319.148980061043, 23500.656687147624, -9923.322063409845, -23105.876609598454, -9868.294502235474, -10360.368559858556, -10452.93773991318, -9847.431121912108, -8143.421939799111, 10415.784601736088, 8395.462702281975, 8940.247849409388, 3930.7112824571445, -9725.226399888432, -3015.7786898521445, -2633.155931874853, -3119.603191592546, -4992.742666232175, -8599.07821944006, 3205.0094609936687, 7267.75779806843, 5497.314717484159, -21711.226070525445, -2950.905357006206, 3381810.398625545, 2708068.110931805, 2047582.610864751, 1378626.9631140158, 681544.6287419036, 136.62548101756045, 699717.3